In [1]:
# load the text

from pathlib import Path

raw_text = Path("../data/the-verdict.txt").read_text(encoding="utf-8")

print("characters:", len(raw_text))
print(raw_text[:100])

characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [2]:
# split into tokens with a regex

import re

# Stage 1: split on whitespace only.
print(re.split(r"(\s)", "Hello, world. Is this-- a test?"))

# Stage 2: also split on commas and periods. The capturing parentheses keep
# the delimiters in the output instead of throwing them away.
print(re.split(r"([,.]|\s)", "Hello, world. Is this-- a test?"))

# Stage 3: the full pattern the book settles on. It separates all common
# punctuation and the double dash as their own tokens.
pattern = r'([,.:;?_!"()\']|--|\s)'
tokens = re.split(pattern, "Hello, world. Is this-- a test?")

# Splitting leaves empty strings and bare whitespace in the list.
# Keep only tokens with actual content.
tokens = [t.strip() for t in tokens if t.strip()]
print(tokens)

['Hello,', ' ', 'world.', ' ', 'Is', ' ', 'this--', ' ', 'a', ' ', 'test?']
['Hello', ',', '', ' ', 'world', '.', '', ' ', 'Is', ' ', 'this--', ' ', 'a', ' ', 'test?']
['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [3]:
# tokenize the whole story

preprocessed = re.split(pattern, raw_text)
preprocessed = [t.strip() for t in preprocessed if t.strip()]

print("tokens:", len(preprocessed))
print(preprocessed[:30])

tokens: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [4]:
# build the vocabulary

# A vocabulary is just a mapping from each unique token string to an integer.
# set() removes duplicates, sorted() makes the numbering deterministic.
all_words = sorted(set(preprocessed))
vocab = {token: idx for idx, token in enumerate(all_words)}

print("vocab size:", len(vocab))

# Peek at the first few entries. Punctuation sorts before letters.
for token, idx in list(vocab.items())[:15]:
    print(idx, repr(token))

vocab size: 1130
0 '!'
1 '"'
2 "'"
3 '('
4 ')'
5 ','
6 '--'
7 '.'
8 ':'
9 ';'
10 '?'
11 'A'
12 'Ah'
13 'Among'
14 'And'


In [5]:
# tokenizer class

class SimpleTokenizerV1:
    def __init__(self, vocab):
        # string -> int, used by encode
        self.str_to_int = vocab
        # int -> string, the reverse mapping, used by decode
        self.int_to_str = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        # Same splitting as before, so text is tokenized the way the
        # vocabulary was built. Then look up each token's integer.
        tokens = re.split(pattern, text)
        tokens = [t.strip() for t in tokens if t.strip()]
        return [self.str_to_int[t] for t in tokens]

    def decode(self, ids):
        # Look up each integer's string and join with spaces.
        text = " ".join(self.int_to_str[i] for i in ids)
        # Joining put a space before every punctuation mark ("word ,").
        # This regex removes the space before punctuation to tidy it up.
        return re.sub(r'\s+([,.?!"()\'])', r"\1", text)

In [6]:
# round-trip a sentence from the story

tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know, "Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [7]:
# break it

# "Hello" never appears in the story, so it is not in the vocabulary
try:
    tokenizer.encode("Hello, do you like tea?")
except KeyError as e:
    print(f"KeyError: no vocabulary entry for {e}")

KeyError: no vocabulary entry for 'Hello'


In [8]:
# extend the vocabulary with special tokens

# Two special tokens the book adds:
#   <|unk|>       stands in for any word not in the vocabulary
#   <|endoftext|> marks a boundary between unrelated documents, so the
#                 model learns that text after it is a fresh start
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: idx for idx, token in enumerate(all_tokens)}

print("vocab size:", len(vocab))
# They got the last two ids.
for token, idx in list(vocab.items())[-3:]:
    print(idx, repr(token))

vocab size: 1132
1129 'yourself'
1130 '<|endoftext|>'
1131 '<|unk|>'


In [9]:
# tokenizer V2, which never raises

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        tokens = re.split(pattern, text)
        tokens = [t.strip() for t in tokens if t.strip()]
        # The one change from V1: unknown words map to <|unk|> instead of
        # raising a KeyError.
        tokens = [t if t in self.str_to_int else "<|unk|>" for t in tokens]
        return [self.str_to_int[t] for t in tokens]

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r'\s+([,.?!"()\'])', r"\1", text)


tokenizer = SimpleTokenizerV2(vocab)

# Two unrelated sentences joined by the end-of-text marker.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join([text1, text2])

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


# BPE with tiktoken

In [10]:
# switch to BPE with tiktoken

import tiktoken

# The exact tokenizer OpenAI trained for GPT-2. 50,257 entries.
tokenizer = tiktoken.get_encoding("gpt2")
print("vocab size:", tokenizer.n_vocab)

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."

# allowed_special is needed because tiktoken refuses special tokens in
# input by default, to stop user text from injecting them by accident.
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(ids)

decoded = tokenizer.decode(ids)
print(decoded)

# The check the simple tokenizer could not pass.
print("exact round-trip:", decoded == text)

vocab size: 50257
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.
exact round-trip: True


In [11]:
len(tokenizer.encode(raw_text))

5145

In [12]:
tokenizer.encode(raw_text)[:10]

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]

In [13]:
# see how BPE handles an unknown word

ids = tokenizer.encode("Akwirw ier")
print(ids)

# Decode each id on its own to see the pieces.
for i in ids:
    print(i, repr(tokenizer.decode([i])))

print(tokenizer.decode(ids))

[33901, 86, 343, 86, 220, 959]
33901 'Ak'
86 'w'
343 'ir'
86 'w'
220 ' '
959 'ier'
Akwirw ier


In [14]:
single = 0
multi = []
for word in vocab:
    # Encode with a leading space, since that is how a word appears
    # mid-sentence and how GPT-2's vocabulary stores most words.
    n = len(tokenizer.encode((" " + word), allowed_special={"<|endoftext|>"}))
    if n == 1:
        single += 1
    else:
        multi.append((word, n))

print(f"{single} of {len(vocab)} regex tokens are a single tiktoken token")
print("some multi-token words:", multi[:15])

927 of 1132 regex tokens are a single tiktoken token
some multi-token words: [('Arrt', 2), ('Croft', 2), ('Destroyed', 2), ('Devonshire', 3), ('Dubarry', 2), ('Emperors', 2), ('Gisburn', 3), ('Gisburns', 4), ('Grafton', 3), ('Grindle', 2), ('Grindles', 2), ('HAD', 2), ('Hermia', 2), ('Jove', 2), ('Moon-dancers', 4)]


In [15]:
for s in ["ier", " ier", "Hello", " Hello", "Hello world", "Hello  world"]:
    ids = tokenizer.encode(s)
    print(repr(s), "->", ids, [tokenizer.decode([i]) for i in ids])

'ier' -> [959] ['ier']
' ier' -> [220, 959] [' ', 'ier']
'Hello' -> [15496] ['Hello']
' Hello' -> [18435] [' Hello']
'Hello world' -> [15496, 995] ['Hello', ' world']
'Hello  world' -> [15496, 220, 995] ['Hello', ' ', ' world']


# sliding-window dataset

In [16]:
import torch
from torch.utils.data import Dataset, DataLoader

In [17]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the whole text once.
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Slide a window of max_length over the ids, moving by stride each
        # step. Stop when the window would run off the end. The +1 on the
        # target slice is the shift: target[j] is the token after input[j].
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # A Dataset needs exactly these two methods.
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [18]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    # DataLoader stacks individual items into batches and handles shuffling.
    # drop_last discards a final partial batch so every batch has the same shape.
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)

In [19]:
raw_text[:5], len(raw_text)

('I HAD', 20479)

In [20]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
dataloader

In [21]:
data_iter = iter(dataloader)
data_iter

In [22]:
data_iter = iter(dataloader)
first = next(data_iter)
second = next(data_iter)
print("first batch: ", first)
print("second batch:", second)
print(first[0].shape)

first batch:  [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
second batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
torch.Size([1, 4])


In [23]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4,
                                  stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))
print("inputs shape:", inputs.shape)   # expect (8, 4): batch_size, max_length
print("targets shape:", targets.shape)

inputs shape: torch.Size([8, 4])
targets shape: torch.Size([8, 4])


In [24]:
inputs

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

In [25]:
targets

tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

### Token embeddings

In [26]:
# An nn.Embedding is a lookup table: a (vocab_size, output_dim) matrix of
# learnable numbers. Indexing it with a token id returns that row. No math.
vocab_size = 50257     # GPT-2's vocabulary
output_dim = 256       # the book uses 256 here; real GPT-2 small uses 768

In [27]:
torch.manual_seed(123)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(token_embedding_layer.weight.shape)

torch.Size([50257, 256])


In [28]:
# Every id in the (8, 4) batch is replaced by its 256-number row.
token_embeddings = token_embedding_layer(inputs)
print("token embeddings:", token_embeddings.shape)   # (8, 4, 256)

# Prove it is a lookup: the output for id 40 is row 40 of the weight matrix.
print(torch.equal(token_embedding_layer(torch.tensor([40]))[0],
                  token_embedding_layer.weight[40]))   # True

token embeddings: torch.Size([8, 4, 256])
True


### Positional embeddings

In [29]:
# Attention has no idea which token came first; it sees a set, not a sequence.
# So add a second lookup table, one row per POSITION, onto the token vectors.
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [30]:
# Positions are 0, 1, 2, 3, the same for every sequence in the batch.
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("pos embeddings:", pos_embeddings.shape)   # (4, 256)

pos embeddings: torch.Size([4, 256])


In [31]:
# (8, 4, 256) + (4, 256): broadcasting adds the same 4 position rows to
# every one of the 8 sequences.
input_embeddings = token_embeddings + pos_embeddings
print("input embeddings:", input_embeddings.shape)   # (8, 4, 256)

input embeddings: torch.Size([8, 4, 256])
